In [28]:
#importing libraries

import numpy as np
import pandas as pd
import pickle
import tensorflow as tf

from tensorflow.keras.models import load_model


In [29]:
#loading saved model, scaler object and encoders

model = load_model("model.keras")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,837 (34.52 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,892 (23.02 KB)

In [30]:
with open("standard_scaler.pkl", "rb") as file_obj:
    scaler = pickle.load(file_obj)

with open("onehot_encoder.pkl", "rb") as file_obj:
    onehot_encoder = pickle.load(file_obj)   

with open("label_encoder.pkl", "rb") as file_obj:
    label_encoder = pickle.load(file_obj)        

In [31]:
#sample input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary': 50000
}


In [32]:
#encoding categorical features
df = pd.DataFrame(data=input_data, index=[0])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [33]:
df["Gender"]=label_encoder.transform(df["Gender"])
df["Gender"]

0    1
Name: Gender, dtype: int64

In [35]:
geo_arr = onehot_encoder.transform(df[['Geography']]).toarray()
geo_df = pd.DataFrame(geo_arr, columns=onehot_encoder.get_feature_names_out())
geo_df.reset_index(drop=True)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [ ]:
df = pd.concat([df.reset_index(drop=True), geo_df], axis=1)

In [38]:
df.drop(columns=['Geography'], inplace=True)
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [39]:
#scaling the data

input_arr = scaler.transform(df)
input_arr

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [53]:
#prediction on input data

prediction =model.predict(input_arr)[0][0]

if prediction > 0.5:
    print(f"Prediction is {prediction:.2f}---Customer is likely to churn")
else:
    print(f"Prediction is {prediction:.2f}---Customer is not likely to churn")    


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction is 0.03---Customer is not likely to churn
